# FHB Pixel Classification — Threshold, GMM, Bayes GMM, SVM & Logistic Regression

**Dataset:** `Pixels_Test.xlsx` (held-out evaluation set for all methods) | `Pixels_Train.xlsx` (training/initialisation set for Methods 2–5)  
**Ground truth:** `Cluster` column — 0 = Healthy, 1 = FHB Infected  

**Sections:**
1. Normalised R/(R+G) threshold at 0.495
2. GMM initialised from `precisions_init.csv` / `means_init.csv` (both derived from Pixel Values), fit on Pixel Values
3. Per-class GMM with Bayes rule (RGB)
4. Support vector classifier (RGB)
5. Logistic regression (RGB)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.mixture import GaussianMixture
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.svm import SVC

np.random.seed(42)

## 1. Load Data

In [ ]:
df_test = pd.read_excel('data\Pixels_Test.xlsx', sheet_name='Sheet1') #change path to your local path
df_vals = pd.read_excel('data\Pixels_Training.xlsx', sheet_name='Sheet1') #change path to your local path

means_df   = pd.read_csv('data\means.csv', index_col=0) #change path to your local path
prec_df    = pd.read_csv('data\precisions.csv', index_col=0) #change path to your local path

print('Pixels_Test shape :', df_test.shape)
print('Pixel Values shape:', df_vals.shape)
print('Class balance (Pixels_Test):')
print(df_test['Cluster'].value_counts().rename({0:'Healthy',1:'Infected'}))
print('\nClass balance (Pixel Values):')
print(df_vals['Cluster'].value_counts().rename({0:'Healthy',1:'Infected'}))

In [ ]:
# Feature matrices and ground-truth labels
X_test  = df_test[['R','G','B']].values.astype(np.float64)
y_test  = df_test['Cluster'].values

X_vals  = df_vals[['R','G','B']].values.astype(np.float64)
y_vals  = df_vals['Cluster'].values

# GMM initialisations from CSV
means_init     = means_df[['R','G','B']].values.astype(np.float64)   # (2,3) — row 0 Healthy, row 1 Infected
prec_flat      = prec_df[['R_R','R_G','R_B','G_R','G_G','G_B','B_R','B_G','B_B']].values.astype(np.float64)
precisions_init = prec_flat.reshape(2, 3, 3)                           # (2,3,3)

print('means_init:\n', means_df)
print('\nprecisions_init shape:', precisions_init.shape)
# Spectral index for Pixel Values (used by Methods 3, 8 for density fitting)
si_vals = X_vals[:, 0] / (X_vals[:, 0] + X_vals[:, 1])


---
## Helper — Evaluation Report

In [ ]:
def evaluate(y_true, y_pred, title='', ax=None):
    """Print precision/recall/F1 and plot confusion matrix."""
    p  = precision_score(y_true, y_pred, pos_label=1)
    r  = recall_score(y_true,    y_pred, pos_label=1)
    f1 = f1_score(y_true,        y_pred, pos_label=1)
    cm = confusion_matrix(y_true, y_pred)

    print(f'\n{"─"*50}')
    print(f'  {title}')
    print(f'  Precision : {p:.4f}')
    print(f'  Recall    : {r:.4f}')
    print(f'  F1 score  : {f1:.4f}')
    print(f'  Confusion matrix (rows=true, cols=pred):')
    print(f'    {cm}')
    print(f'{"─"*50}')

    if ax is not None:
        disp = ConfusionMatrixDisplay(cm, display_labels=['Healthy','Infected'])
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f'{title}\nP={p:.3f}  R={r:.3f}  F1={f1:.3f}', fontsize=10)

    return {'precision': p, 'recall': r, 'f1': f1, 'cm': cm}

---
## Method 1 — Normalized R/(R+G) Threshold   
Pixels with index > 0.495 are classified as **Infected**, ≤ 0.495 as **Healthy**.

In [ ]:
THRESHOLD = 0.495

R_test, G_test = X_test[:, 0], X_test[:, 1]
rg_index_test  = R_test / (R_test + G_test)   # r/(R+G)

y_thresh = (rg_index_test > THRESHOLD).astype(int)

print('Threshold predictions:', np.bincount(y_thresh, minlength=2), '(0=Healthy, 1=Infected)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Method 1 — R/(R+G) Threshold at 0.495', fontsize=13, fontweight='bold')

# Distribution plot
ax = axes[0]
for cls, label, color in [(0,'Healthy','#2196F3'), (1,'Infected','#F44336')]:
    ax.hist(rg_index_test[y_test == cls], bins=40, alpha=0.65,
            label=label, color=color, edgecolor='none')
ax.axvline(THRESHOLD, color='black', lw=2, ls='--', label=f'Threshold = {THRESHOLD}')
ax.set_xlabel('r / (R+G)')
ax.set_ylabel('Count')
ax.set_title('Index Distribution by True Class')
ax.legend()

res_thresh = evaluate(y_test, y_thresh, title='Method 1: R/(R+G) > 0.495', ax=axes[1])
plt.tight_layout()
plt.show()

---
## Method 2 — GMM Initialised from `precisions_init` / `means_init`

`means_init` / `precisions_init` are the per-class RGB means and precision matrices computed from **Pixel Values.xlsx** (confirmed by matching `means_init` against the Pixel Values groupby-`Cluster` means).  
A 2-component full-covariance GMM is initialised from these and fit on **Pixel Values** (training set), then evaluated on **Pixels_Test** (held-out set) — consistent with Methods 3–5.

In [ ]:
gmm2 = GaussianMixture(
    n_components=2,
    covariance_type='full',
    means_init=means_init,
    precisions_init=precisions_init,
    random_state=42,
    tol=1e-6,
    max_iter=500
)
gmm2.fit(X_vals)

print('Fitted means:\n', pd.DataFrame(gmm2.means_, columns=['R','G','B'],
                                       index=['Component 0','Component 1']))

In [ ]:
# Map GMM components to Healthy / Infected by comparing fitted means to means_init
d0_healthy  = np.linalg.norm(gmm2.means_[0] - means_init[0])
d0_infected = np.linalg.norm(gmm2.means_[0] - means_init[1])
INFECTED_COMP = 1 if d0_healthy < d0_infected else 0
print(f'GMM component {INFECTED_COMP} mapped to Infected')

raw_labels = gmm2.predict(X_test)
# Remap so that 1 = Infected in our predictions
if INFECTED_COMP == 0:
    y_gmm2 = 1 - raw_labels
else:
    y_gmm2 = raw_labels

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
fig.suptitle('Method 2 — GMM (precisions_init / means_init)\n'
             'Trained on Pixel Values | Tested on Pixels_Test',
             fontsize=12, fontweight='bold')
res_gmm2 = evaluate(y_test, y_gmm2, title='Method 2: Pre-initialised GMM', ax=ax)
plt.tight_layout()
plt.show()

---
## Method 3 - Per-Class GMM with Bayes Rule

A separate GMM is fit as a **density estimator** for each class.  
At prediction time Bayes rule is applied manually with equal priors (0.5/0.5):

    log_post_k = log p(x | k) + log prior_k
    y_pred = argmax_k log_post_k

This is a generative classifier: each GMM models the within-class RGB distribution.

In [ ]:
PRIOR = 0.5

# Fit density estimators on Pixel Values, apply to Pixels_Test
gmm3_0 = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
gmm3_1 = GaussianMixture(n_components=2, covariance_type='full', random_state=42)

gmm3_0.fit(X_vals[y_vals == 0])   # density model for Healthy (Pixel Values)
gmm3_1.fit(X_vals[y_vals == 1])   # density model for FHB Infected (Pixel Values)

ll_0 = gmm3_0.score_samples(X_test)
ll_1 = gmm3_1.score_samples(X_test)

log_post_0 = ll_0 + np.log(PRIOR)
log_post_1 = ll_1 + np.log(PRIOR)

y_bayes = (log_post_1 > log_post_0).astype(int)

log_evidence = np.logaddexp(log_post_0, log_post_1)
p_infected   = np.exp(log_post_1 - log_evidence)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Method 3 - Per-Class GMM + Bayes Rule (RGB)\n'
             'Trained on Pixel Values | Tested on Pixels_Test | prior=0.5',
             fontsize=11, fontweight='bold')

ax = axes[0]
for cls, label, color in [(0,'Healthy','#2196F3'), (1,'Infected','#F44336')]:
    ax.hist(p_infected[y_test == cls], bins=40, alpha=0.65,
            label=label, color=color, edgecolor='none')
ax.axvline(0.5, color='black', lw=2, ls='--', label='Decision boundary (0.5)')
ax.set_xlabel('P(Infected | x)')
ax.set_ylabel('Count')
ax.set_title('Posterior Probability Distribution')
ax.legend()

res_bayes = evaluate(y_test, y_bayes, title='Method 3: Per-Class Bayes GMM (RGB)', ax=axes[1])
plt.tight_layout()
plt.show()

---
## Method 4 — Support Vector Classifier (RGB)

Trained on `Pixel Values.xlsx`, evaluated on `Pixels_Test.xlsx`.  
Hyperparameters (`C`, `gamma`, `kernel`) optimised via GridSearchCV with 5-fold stratified CV, scored on F1.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_param_grid = {
    'svm__C':      [0.1, 1, 10, 100],
    'svm__gamma':  ['scale', 'auto', 0.001, 0.01, 0.1],
    'svm__kernel': ['rbf', 'linear'],
}

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(random_state=42)),
])

svm_grid = GridSearchCV(svm_pipeline, svm_param_grid, cv=cv5,
                        scoring='f1', n_jobs=-1, verbose=0)
svm_grid.fit(X_vals, y_vals)

print(f'Best SVM params : {svm_grid.best_params_}')
print(f'Best CV F1      : {svm_grid.best_score_:.4f}')

# Cross-validation summary with best parameters
best_svm = svm_grid.best_params_
svm_final = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(C=best_svm['svm__C'], gamma=best_svm['svm__gamma'],
                   kernel=best_svm['svm__kernel'], random_state=42)),
])

cv_svm = cross_validate(svm_final, X_vals, y_vals, cv=cv5,
                        scoring=['accuracy', 'precision', 'recall', 'f1'])
print(f'\n5-Fold CV on Pixel Values (best params):')
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    s = cv_svm[f'test_{metric}']
    print(f'  {metric:>10}: {s.mean():.4f} ± {s.std():.4f}')

# Fit on full Pixel Values, evaluate on Pixels_Test
svm_final.fit(X_vals, y_vals)
y_svm = svm_final.predict(X_test)

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
fig.suptitle('Method 4 — SVM (RGB)\nTrained on Pixel Values | Tested on Pixels_Test',
             fontsize=11, fontweight='bold')
res_svm = evaluate(y_test, y_svm, title='Method 4: SVM (RGB)', ax=ax)
plt.tight_layout()
plt.show()

---
## Method 5 — Logistic Regression (RGB)

Trained on `Pixel Values.xlsx`, evaluated on `Pixels_Test.xlsx`.  
Regularisation strength `C` optimised via GridSearchCV with 5-fold stratified CV, scored on F1.  
Coefficients show the relative contribution of each RGB channel.

In [ ]:
lr_param_grid = {
    'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100],
}

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42)),
])

lr_grid = GridSearchCV(lr_pipeline, lr_param_grid, cv=cv5,
                       scoring='f1', n_jobs=-1, verbose=0)
lr_grid.fit(X_vals, y_vals)

print(f'Best LR params : {lr_grid.best_params_}')
print(f'Best CV F1     : {lr_grid.best_score_:.4f}')

# Cross-validation summary with best C
best_lr_C = lr_grid.best_params_['logreg__C']
lr_final = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(C=best_lr_C, max_iter=1000, random_state=42)),
])

cv_lr = cross_validate(lr_final, X_vals, y_vals, cv=cv5,
                       scoring=['accuracy', 'precision', 'recall', 'f1'])
print(f'\n5-Fold CV on Pixel Values (best params):')
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    s = cv_lr[f'test_{metric}']
    print(f'  {metric:>10}: {s.mean():.4f} ± {s.std():.4f}')

# Fit on full Pixel Values, evaluate on Pixels_Test
lr_final.fit(X_vals, y_vals)
y_lr = lr_final.predict(X_test)

# Coefficients
coef_lr = pd.DataFrame({
    'Channel':     ['R', 'G', 'B'],
    'Coefficient': lr_final.named_steps['logreg'].coef_[0],
})
print(f'\nLogistic Regression Coefficients (standardised, C={best_lr_C}):')
print(coef_lr.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Method 5 — Logistic Regression (RGB)\nTrained on Pixel Values | Tested on Pixels_Test',
             fontsize=11, fontweight='bold')

ax = axes[0]
colors_lr = ['#E53935' if v > 0 else '#43A047' for v in coef_lr['Coefficient']]
ax.barh(coef_lr['Channel'], coef_lr['Coefficient'], color=colors_lr,
        edgecolor='black', linewidth=0.8)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Standardised Coefficient')
ax.set_title(f'Channel Coefficients (C={best_lr_C})')
for i, row in coef_lr.iterrows():
    ax.text(row['Coefficient'] + 0.01 * np.sign(row['Coefficient']),
            i, f"{row['Coefficient']:.4f}", va='center', fontsize=9)

res_lr = evaluate(y_test, y_lr, title='Method 5: Logistic Regression (RGB)', ax=axes[1])
plt.tight_layout()
plt.show()

---
## Comparison — All Methods

In [ ]:
summary = pd.DataFrame({
    'Method': [
        'M1: R/(R+G) Threshold',
        'M2: Full GMM (RGB)',
        'M3: Per-Class Bayes GMM (RGB)',
        'M4: SVM (RGB)',
        'M5: Logistic Regression (RGB)',
    ],
    'Precision': [res_thresh['precision'], res_gmm2['precision'],
                  res_bayes['precision'],  res_svm['precision'], res_lr['precision']],
    'Recall':    [res_thresh['recall'],    res_gmm2['recall'],
                  res_bayes['recall'],     res_svm['recall'],    res_lr['recall']],
    'F1':        [res_thresh['f1'],        res_gmm2['f1'],
                  res_bayes['f1'],         res_svm['f1'],        res_lr['f1']],
}).set_index('Method')

print('\n=== Summary (all methods evaluated on Pixels_Test) ===')
print(summary.sort_values('F1', ascending=False).to_string(float_format='{:.4f}'.format))

methods_plot = [
    ('M1: Threshold',              res_thresh),
    ('M2: Full GMM (RGB)',         res_gmm2),
    ('M3: Bayes GMM (RGB)',        res_bayes),
    ('M4: SVM (RGB)',              res_svm),
    ('M5: Logistic Reg. (RGB)',    res_lr),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Confusion Matrices — All Methods', fontsize=13, fontweight='bold')
for ax, (name, res) in zip(axes.flat, methods_plot):
    ConfusionMatrixDisplay(res['cm'], display_labels=['Healthy', 'Infected']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(
        f"{name}\nP={res['precision']:.3f}  R={res['recall']:.3f}  F1={res['f1']:.3f}",
        fontsize=10)
for ax in axes.flat[len(methods_plot):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(summary))
width = 0.25
ax.bar(x - width, summary['Precision'], width, label='Precision', color='#1E88E5', alpha=0.85)
ax.bar(x,         summary['Recall'],    width, label='Recall',    color='#43A047', alpha=0.85)
ax.bar(x + width, summary['F1'],        width, label='F1',        color='#E53935', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 — All Methods')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

summary